# Telecom Customer Churn Prediction

## Project Overview
This notebook develops a machine learning model to predict customer churn in a telecommunications company. Customer churn refers to when customers stop doing business with a company. Predicting churn allows businesses to take proactive measures to retain customers.

### Objectives:
- Perform exploratory data analysis on telecom customer data
- Preprocess and prepare data for machine learning
- Build and compare multiple classification models
- Identify the best performing model for churn prediction

---

## 1. Import Required Libraries

We import all necessary libraries for:
- **Data manipulation**: NumPy, Pandas
- **Machine Learning**: Scikit-learn (preprocessing, models, metrics)
- **Model evaluation**: Train-test split, accuracy scoring

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.metrics import accuracy_score,precision_score, f1_score, confusion_matrix, recall_score, classification_report

## 2. Load the Dataset

Loading the Telco Customer Churn dataset which contains customer information and whether they churned or not.

In [2]:
df = pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")

## 3. Exploratory Data Analysis (EDA)

### 3.1 Dataset Shape
First, let's understand the dimensions of our dataset - number of rows (customers) and columns (features).

In [3]:
df.shape

(7043, 21)

### 3.2 View First Few Rows
Examining the first few records to understand the structure and type of data we're working with.

In [4]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### 3.3 Column Names
Display all feature names in the dataset.

In [5]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

### 3.4 Dataset Information
Getting detailed information about:
- Data types of each column
- Non-null counts
- Memory usage

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


### 3.5 Missing Values Check
Identifying any missing values in the dataset that need to be handled.

In [7]:
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


### 3.6 Duplicate Records Check
Checking for any duplicate entries in the dataset.

In [8]:
df.duplicated().sum()

np.int64(0)

### 3.7 Statistical Summary
Getting descriptive statistics for numerical features:
- Count, mean, standard deviation
- Min, max, quartiles

In [9]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


### 3.8 Target Variable Distribution
Analyzing the distribution of the target variable (Churn) to check for class imbalance.

In [10]:
df['Churn'].value_counts()

,count
Churn,
No,5174
Yes,1869


## 4. Data Preprocessing

### 4.1 Remove Unnecessary Columns
Dropping the `customerID` column as it's just an identifier and doesn't provide predictive value.

In [11]:
df.drop(columns=['customerID'],inplace=True)

### 4.2 Verify Column Removal
Confirming that the customerID column has been successfully removed.

In [12]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### 4.3 Handle Data Type Issues
Converting `TotalCharges` to numeric format. Some values might be stored as strings with spaces, which need to be handled and converted to NaN, then filled appropriately.

In [13]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


### 4.4 Encode Target Variable
Converting the `Churn` column from categorical ('Yes'/'No') to binary (1/0) for model training.

In [14]:
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})


## 5. Prepare Data for Modeling

### 5.1 Separate Features and Target
Splitting the dataset into:
- **X**: Features (independent variables)
- **y**: Target variable (Churn)

In [15]:
# Divide the data for train and test
X = df.drop(columns=['Churn'])
y = df['Churn']

### 5.2 Train-Test Split
Dividing the data into training and testing sets:
- **Training set**: Used to train the models
- **Test set**: Used to evaluate model performance on unseen data
- Using 80-20 split with stratification to maintain class distribution

In [16]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42)

## 6. Feature Engineering

### 6.1 Identify Categorical Features
Identifying all categorical columns that need to be encoded for machine learning algorithms.

In [17]:
# Encoding
categorical_features = [
    'gender','Partner','Dependents','PhoneService','MultipleLines',
    'InternetService','OnlineSecurity','OnlineBackup','DeviceProtection',
    'TechSupport','StreamingTV','StreamingMovies','Contract',
    'PaperlessBilling','PaymentMethod'
]

numerical_features = ['tenure','MonthlyCharges','TotalCharges']


### 6.2 Create Preprocessing Pipeline
Setting up a `ColumnTransformer` with `OneHotEncoder` to:
- Convert categorical features into numerical format
- Create binary columns for each category
- Handle unknown categories during prediction

In [18]:
from sklearn.preprocessing import OneHotEncoder
categorical_cols = OneHotEncoder(drop='first',handle_unknown='ignore')
numerical_cols = StandardScaler()

In [19]:
preprocessor=ColumnTransformer(
    transformers=[
    ('num',numerical_cols,numerical_features),
    ('cat',categorical_cols,categorical_features)
    ]
)

## 7. Model Development and Evaluation

### 7.1 Define Multiple Classification Models
Creating a dictionary of different machine learning algorithms to compare:
- **Logistic Regression**: Linear model for binary classification
- **Decision Tree**: Tree-based model that learns decision rules
- **Random Forest**: Ensemble of decision trees
- **Support Vector Machine (SVM)**: Finds optimal hyperplane for classification
- **K-Nearest Neighbors (KNN)**: Classifies based on nearest training examples
- **Naive Bayes**: Probabilistic classifier based on Bayes' theorem

In [20]:
# Create mutlitple models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

### 7.2 Train and Evaluate All Models
For each model:
1. Create a pipeline combining preprocessing and the model
2. Train on the training data
3. Make predictions on the test data
4. Calculate accuracy score
5. Store results for comparison

In [21]:
models = {
    "Logistic Regression":LogisticRegression(max_iter=1000),
    "Random Forest":RandomForestClassifier(n_estimators=100,random_state=42),
    "Decision Tree":DecisionTreeClassifier(random_state=42,max_depth=5),
    "XGBoost":XGBClassifier(n_estimators=200,learning_rate=0.05,max_depth=4,eval_metric='logloss',random_state=42)
}

### 7.3 Compare Model Performance
Displaying all model results sorted by accuracy to identify the best performing model.

In [22]:
results = []

for model_name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    })


## 8. Results and Conclusions

### Model Performance Summary
The table above shows the accuracy of each model on the test dataset. 

### Key Findings:
- We successfully built and compared 6 different classification models
- All models were evaluated using the same train-test split for fair comparison
- The model with the highest accuracy is the best candidate for predicting customer churn

### Next Steps:
- Fine-tune the best performing model using hyperparameter optimization
- Analyze feature importance to understand key churn drivers
- Consider additional metrics (precision, recall, F1-score) for imbalanced data
- Implement cross-validation for more robust performance estimates
- Deploy the model for real-time churn prediction

In [23]:
results_df = pd.DataFrame(results).sort_values(by='Accuracy',ascending=False)
results_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.819730,0.679758,0.603217,0.639205
3,XGBoost,0.812633,0.676375,0.560322,0.612903
2,Decision Tree,0.806246,0.704918,0.461126,0.557536
1,Random Forest,0.786373,0.633333,0.458445,0.531882
